In [1]:
from pyspark.sql import SparkSession
import math

spark = SparkSession.builder.appName("NaiveBayes").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/15 14:10:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/10/15 14:10:36 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
def is_transpose(cols):
    """
        Función para detectar que
        un csv está volteado

        status: not working xD

        solo detecta si el csv lo dejan como
        ,1,2,3,4...

        es decir que una columna no tiene nombre
    """
    count_not_valid = 0

    for col in cols:
        if col[0] == "_":
            count_not_valid += 1
        
    return True if count_not_valid == 1 else False

def read_csv():
    try:
        #path = input("Ingresa el path del archivo csv: ")
        path = "naive_data.csv"

        df = spark.read.csv(path, inferSchema=True, header=True)

        cols = df.columns

        if is_transpose(cols):
            df = df.transpose()
            df.show()

        #print(df.show())

    except Exception as e:
        print(f'Error leyendo el archivo')
        print(str(e))
    else:
        return df, cols

In [3]:
df, cols = read_csv()

df.show()

+--------+----+--------+------+----+
| Outlook|Temp|Humidity|  Wind|Play|
+--------+----+--------+------+----+
|   Sunny| Hot|    High|  Weak|  No|
|   Sunny| Hot|    High|Strong|  No|
|Overcast| Hot|    High|  Weak| Yes|
|    Rain|Mild|    High|  Weak| Yes|
|    Rain|Cool|  Normal|  Weak| Yes|
|    Rain|Cool|  Normal|Strong|  No|
|Overcast|Cool|  Normal|Strong| Yes|
|   Sunny|Mild|    High|  Weak|  No|
|   Sunny|Cool|  Normal|  Weak| Yes|
|    Rain|Mild|  Normal|  Weak| Yes|
+--------+----+--------+------+----+



In [4]:
target_col = input(f'Selecciona la columna objetivo\n{cols}: ')
cols.remove(target_col)

In [5]:
def get_prediction_vect(cols):
    print("Construye el vector de predicción")
    vector = {}

    for col in cols:
        unicos = [row[col] for row in df.select(col).distinct().collect()]

        print(f"Columna {col}: valores únicos -> {unicos}")
        option = input(f"Escoge un valor para '{col}' de la lista anterior: ")

        if not option == "":
            vector[col] = option

    print("\nVector de predicción construido:")
    
    return vector

vector_prediccion = get_prediction_vect(cols)

print(vector_prediccion)

Construye el vector de predicción
Columna Outlook: valores únicos -> ['Sunny', 'Rain', 'Overcast']
Columna Temp: valores únicos -> ['Cool', 'Mild', 'Hot']
Columna Humidity: valores únicos -> ['High', 'Normal']
Columna Wind: valores únicos -> ['Strong', 'Weak']

Vector de predicción construido:
{'Outlook': 'Sunny', 'Temp': 'Cool', 'Humidity': 'High', 'Wind': 'Weak'}


# 1. A Priori

In [6]:
# Probabilidad A Priori
# 1. Definir el conjunto de clases C

classes = [val[f'{target_col}'] for val in df.select(df[f'{target_col}']).distinct().collect()]

classes

['No', 'Yes']

In [7]:
# 2. Calcular probabilidades a priori para cada uno de las diferentes clases
a_priori_probs = {}

n = df.count()

for class_val in classes:
    count = df.filter(df[f'{target_col}'] == class_val).count()
    print(f'Existen {count} registros donde {target_col} = {class_val}')

    a_priori_probs[class_val] = count/n

a_priori_probs

Existen 4 registros donde Play = No
Existen 6 registros donde Play = Yes


{'No': 0.4, 'Yes': 0.6}

# 2. Likelihoods

In [8]:
cols

['Outlook', 'Temp', 'Humidity', 'Wind']

In [9]:
# Inicializando diccionario de likelihoods
likelihoods = {class_val:[] for class_val in classes}

for col, val in vector_prediccion.items():
    print(f'#### {col} = {val}')
    df_grouped = df.groupBy(f'{col}', f'{target_col}').count()

    # Tomar las filas donde se cumpla COL == VAL
    df_grouped = df_grouped.filter(df_grouped[f'{col}'] == f'{val}')

    for class_val in classes:
        # Contar registros por clase
        count_class = df.filter(df[f'{target_col}'] == class_val).count()

        # Tomar la fila donde TARGET_COL == CLASS_i
        df_temp = df_grouped.filter(df_grouped[f'{target_col}'] == f'{class_val}')

        if df_temp.count() == 0:
            print(f'No hay registros para ({col}={val} | {class_val})')
            continue

        # Seleccionar la columna target para obtener el conteo de registros
        count_grouped = df_temp.select('count').collect()[0]['count']

        print(f'Para ({col}={val} | {class_val}) hay {count_grouped} registros')

        # Calcular likelihood
        likelihoods[class_val].append(count_grouped / count_class)

#### Outlook = Sunny
Para (Outlook=Sunny | No) hay 3 registros
Para (Outlook=Sunny | Yes) hay 1 registros
#### Temp = Cool
Para (Temp=Cool | No) hay 1 registros
Para (Temp=Cool | Yes) hay 3 registros
#### Humidity = High
Para (Humidity=High | No) hay 3 registros
Para (Humidity=High | Yes) hay 2 registros
#### Wind = Weak
Para (Wind=Weak | No) hay 2 registros
Para (Wind=Weak | Yes) hay 5 registros


In [10]:
likelihoods

{'No': [0.75, 0.25, 0.75, 0.5],
 'Yes': [0.16666666666666666, 0.5, 0.3333333333333333, 0.8333333333333334]}

Coomparación

In [11]:
final = {}
for class_val, probs in likelihoods.items():
    product = math.prod(probs)
    final[class_val] = product

prediccion = max(final, key=final.get)

print("Categoría con mayor probabilidad:", prediccion)
print("Probabilidad:", final[prediccion])


Categoría con mayor probabilidad: No
Probabilidad: 0.0703125
